# ✂️ U-Bit-SegNet: 1.58-bit 最強背景削除モデル学習
このノートブックでは、超省メモリな 1.58-bit双方向SSMを用いた画像セグメンテーションモデルの学習を行います。
高品質な **DIS5Kデータセット** を使用し、Colabのタイムアウト対策として Google Drive を活用した最強の学習ワークフローを提供します。

## 1. 📂 Google Drive のマウント
学習済みのモデルや、重いデータセットを退避させるために Google Drive をマウントします。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. ⚙️ ライブラリのインストールと最新コードの取得

In [ ]:
!pip install -q torch torchvision transformers tqdm pillow datasets
![ -d 'BitMC-SSM' ] || git clone https://github.com/fukayatti/BitMC-SSM.git
%cd BitMC-SSM
!git pull origin main

## 3. 📦 【初回のみ】DIS5Kデータの解凍と事前リサイズ
**※すでにGoogle Driveに `dis5k_256.zip` を作成済みの場合は、このセルはスキップしてください。**

Drive上にある未リサイズの巨大なデータ（`dis5k.zip`）をローカルに持ってきて解凍し、あらかじめ `256x256` にリサイズしてから、再度Zip化して Google Drive に退避させます。

In [ ]:
# 1. 巨大な未リサイズのZipをローカルにコピーして解凍
!cp /content/drive/MyDrive/BitMC-SSM/dis5k.zip /content/
!unzip -q -o /content/dis5k.zip -d /

# 2. CPU負荷対策として全画像を256x256に事前リサイズ
!python python/resize_dataset.py --data_dir "/content/data/dis5k" --out_dir "/content/data/dis5k_256" --size 256

# 3. 次回以降一瞬で読み込めるように、リサイズ済みのものをZip化してGoogle Driveへ退避
!zip -q -r /content/drive/MyDrive/BitMC-SSM/dis5k_256.zip /content/data/dis5k_256
print("✅ 初回セットアップ完了！Driveに軽量な dis5k_256.zip を保存しました。")

## 4. 🚀 【毎回実行】Driveからデータを一瞬で復元
Colabのセッションがリセットされた際も、Driveに保存したZipを展開するだけで、ネットワークダウンロードやリサイズをスキップして即座に学習環境を復元できます。

In [ ]:
!cp /content/drive/MyDrive/BitMC-SSM/dis5k_256.zip /content/
!unzip -q -o /content/dis5k_256.zip -d /
print("✅ データの復元が完了しました。学習を開始できます。")

## 5. 🧠 U-Bit-SegNet の学習開始
すでに途中まで学習したモデルが Drive にある場合は、`--resume` に自動でパスが渡されて途中から再開します。
初めて学習する場合は `--resume` にファイルが見つからなくても自動で最初からスタートします。

In [ ]:
!python python/train_segnet.py \
    --data_dir "/content/data/dis5k_256" \
    --output_dir "/content/drive/MyDrive/BitMC-SSM/checkpoints_dis5k" \
    --resume "/content/drive/MyDrive/BitMC-SSM/checkpoints_dis5k/bit_segnet_best.pt" \
    --img_size 256 \
    --batch_size 64 \
    --epochs 50 \
    --base_dim 32 \
    --depths "1,1,2,1"

## 6. 🔮 推論テスト (画像の背景切り抜き)
学習した最高性能のチェックポイント(`bit_segnet_best.pt`)を使って、画像を切り抜きます。

In [ ]:
from IPython.display import Image, display

!python python/infer_segnet.py \
    --image "/content/data/dis5k_256/val/images/1.jpg" \
    --checkpoint "/content/drive/MyDrive/BitMC-SSM/checkpoints_dis5k/bit_segnet_best.pt" \
    --output "result.png"

display(Image("result.png"))